<a href="https://colab.research.google.com/github/M8-12NJ/AI_Codingal/blob/main/Multi_tool_AI_app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AI Teaching Assistant + Math Mastermind  (single Colab cell)
# Colab Secrets needed (🔑 icon in left sidebar, enable notebook access):
#   GROQ_API_KEY   -> https://console.groq.com
#   HF_API_KEY     -> https://huggingface.co/settings/tokens   (optional)
# ============================================================
import os, re, subprocess, sys, time
from google.colab import userdata

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "streamlit", "openai", "huggingface_hub",
                "diffusers", "transformers", "accelerate", "safetensors"], check=True)

def _secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

# Keys are passed to the Streamlit process via environment variables
# (userdata only works inside the notebook kernel, not in subprocesses).
ENV = os.environ.copy()
ENV["GROQ_API_KEY"] = _secret("GROQ_API_KEY")
ENV["HF_API_KEY"] = _secret("HF_API_KEY")
if not ENV["GROQ_API_KEY"] and not ENV["HF_API_KEY"]:
    print("⚠️ No keys found. Add GROQ_API_KEY and/or HF_API_KEY in Colab Secrets.")

APP_CODE = r'''
import io, os, re
import streamlit as st
from openai import OpenAI
from huggingface_hub import InferenceClient

# ---------------- Provider config (from env, set by Colab userdata) ----------------
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
HF_API_KEY = os.environ.get("HF_API_KEY", "")
GROQ_URL = "https://api.groq.com/openai/v1"
GROQ_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]  # fallback only
GROQ_PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
                  "openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3-32b"]
GROQ_SKIP = ("whisper", "tts", "guard", "orpheus", "playai", "embed")
HF_MODELS = ["meta-llama/Llama-3.1-8B-Instruct"]

def groq_response(prompt, temperature, max_tokens):
    if not GROQ_API_KEY:
        return "Error: GROQ_API_KEY missing in Colab Secrets."
    c = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_URL)

    # Ask Groq which models this key can actually use (cached per session)
    if "groq_models" not in st.session_state:
        try:
            ids = [m.id for m in c.models.list().data]
            chat = [i for i in ids if not any(s in i.lower() for s in GROQ_SKIP)]
            ordered = [m for m in GROQ_PREFERRED if m in chat] + [m for m in chat if m not in GROQ_PREFERRED]
            st.session_state["groq_models"] = ordered or GROQ_MODELS
        except Exception as e:
            return (f"Could not list Groq models - the API key is probably invalid or blocked.\n"
                    f"Details: {type(e).__name__}: {e}")
    models = st.session_state["groq_models"]

    errs = []
    for m in models[:5]:
        try:
            r = c.chat.completions.create(
                model=m,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            errs.append(f"{m}: {type(e).__name__}: {e}")
    st.session_state.pop("groq_models", None)  # re-discover next time
    return ("Groq request failed for all models.\n" + "\n".join(errs) +
            "\nFix: check your key/project limits at console.groq.com, or switch provider in the sidebar.")

def hf_response(prompt, temperature, max_tokens):
    if not HF_API_KEY:
        return "Error: HF_API_KEY missing in Colab Secrets."
    last_err = None
    for m in HF_MODELS:
        try:
            c = InferenceClient(model=m, token=HF_API_KEY)
            r = c.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            last_err = e
    return (f"Hugging Face model failed.\nTried models: {HF_MODELS}\n"
            f"Fix: switch provider to Groq in the sidebar, or edit HF_MODELS.\n"
            f"Details: {type(last_err).__name__}: {last_err}")

def generate_response(prompt, temperature=0.3, max_tokens=512):
    provider = st.session_state.get("provider", "Groq")
    fn = groq_response if provider == "Groq" else hf_response
    return fn(prompt, temperature, max_tokens)

# ---------------- Math rendering helpers ----------------
# Matches $$...$$ and $...$ so math segments can be left untouched.
_MATH_RE = re.compile(r"(\$\$.+?\$\$|\$[^$\n]+?\$)", re.S)

def _protect_ops(t):
    """Escape operator characters Markdown would eat, in NON-math text only."""
    # 6871627 * 1287, 6871627*1287, 6871627 *1287
    t = re.sub(r"(?<=\d)[ ]?(?<!\*)\*(?!\*)[ ]?(?=\d)", r" \\* ", t)
    # x * y (spaces on both sides; real italics never look like this)
    t = re.sub(r"(?<=[\w)])(?<!\*) \*(?!\*) (?=[\w(])", r" \\* ", t)
    # a lone ~ would start strikethrough
    t = t.replace("~", r"\~")
    return t

def fix_math(text):
    """Normalize LaTeX delimiters to $...$ / $$...$$."""
    text = re.sub(r"\\\[(.+?)\\\]", lambda m: "$$" + m.group(1).strip() + "$$", text, flags=re.S)
    text = re.sub(r"\\\((.+?)\\\)", lambda m: "$" + m.group(1).strip() + "$", text, flags=re.S)
    # Some models drop the backslashes and leave bare [ ... ] blocks on their own lines
    text = re.sub(r"^\s*\[\s*\n(.+?)\n\s*\]\s*$", lambda m: "$$" + m.group(1).strip() + "$$",
                  text, flags=re.S | re.M)
    return text

_DISPLAY_RE = re.compile(r"\$\$(.+?)\$\$", re.S)

def _protect_prose(t):
    """Escape operators in prose, leaving inline $...$ math untouched."""
    parts = _MATH_RE.split(t)
    for i in range(0, len(parts), 2):
        parts[i] = _protect_ops(parts[i])
    return "".join(parts)

def render_answer(text):
    """Prose -> st.markdown, display math ($$...$$) -> st.latex (never touched by Markdown)."""
    text = fix_math(text)
    pos = 0
    for m in _DISPLAY_RE.finditer(text):
        prose = text[pos:m.start()]
        if prose.strip():
            st.markdown(_protect_prose(prose))
        st.latex(m.group(1).strip())
        pos = m.end()
    tail = text[pos:]
    if tail.strip():
        st.markdown(_protect_prose(tail))

def esc_md(s):
    """Show user text literally: escape inline syntax, line-start operators, keep line breaks."""
    s = re.sub(r"([\\`*_~$|<>])", r"\\\1", str(s))
    s = re.sub(r"(?m)^(\s*)([+\-#=])", r"\1\\\2", s)      # + - # = at line start
    s = re.sub(r"(?m)^(\s*\d+)([.)])", r"\1\\\2", s)      # "1." / "1)" list markers
    return s.replace("\n", "  \n")

# ---------------- UI ----------------
MATH_SYSTEM = """You are a Math Mastermind.
Solve with clear step-by-step reasoning, correct notation, and a final answer.
Verify when possible; mention an alternative method briefly if relevant.
Format all math in LaTeX using $...$ for inline and $$...$$ for display equations.
Write every calculation inside LaTeX (e.g. $6871627 \\times 1287$), never as plain text with * or x.
Never use \\( \\) or \\[ \\] delimiters, and never put bare [ ] or ( ) around equations."""

TEACH_SYSTEM = """You are a helpful AI Teaching Assistant.
Explain clearly and concisely. If you write any math, format it in LaTeX using
$...$ for inline and $$...$$ for display equations. Never use \\( \\) or \\[ \\] delimiters.
Write every calculation inside LaTeX (e.g. $6871627 \\times 1287$), never as plain text with * or x."""

def export_txt(history):
    txt = "".join([f"Q{i}: {h['question']}\nA{i}: {h['answer']}\n\n" for i, h in enumerate(history, 1)])
    bio = io.BytesIO(txt.encode("utf-8")); bio.seek(0); return bio

def teaching_answer(q):
    return generate_response(f"{TEACH_SYSTEM}\n\nQuestion: {q}", temperature=0.3, max_tokens=1024)

def math_answer(q, level):
    prompt = f"{MATH_SYSTEM}\n\nDifficulty: {level}\nMath Problem: {q}"
    return generate_response(prompt, temperature=0.1, max_tokens=1024)

def run_ai_teaching_assistant():
    st.title("🤖 AI Teaching Assistant")
    st.session_state.setdefault("history_ata", [])
    c1, c2 = st.columns([1, 2])
    if c1.button("🧹 Clear", key="c_ata"):
        st.session_state.history_ata = []; st.rerun()
    if st.session_state.history_ata:
        c2.download_button("📄 Export", export_txt(st.session_state.history_ata),
                           "AI_Teaching_Assistant_Conversation.txt", "text/plain")
    q = st.text_input("Enter your question:", key="q_ata")
    if st.button("Ask", key="a_ata"):
        if not q.strip():
            st.warning("⚠️ Enter a question.")
        else:
            with st.spinner("Thinking..."):
                st.session_state.history_ata.append({"question": q.strip(), "answer": teaching_answer(q.strip())})
            st.rerun()

    if not st.session_state.history_ata: return
    with st.container(height=600):
        for i, qa in enumerate(st.session_state.history_ata, 1):
            with st.container(border=True):
                st.markdown(f"**Q{i}:** {esc_md(qa['question'])}")
                render_answer(qa["answer"])

def run_math_mastermind():
    st.title("🧮 Math Mastermind")
    st.session_state.setdefault("history_mm", [])
    st.session_state.setdefault("k_mm", 0)
    c1, c2 = st.columns([1, 2])
    if c1.button("🧹 Clear", key="c_mm"):
        st.session_state.history_mm = []; st.rerun()
    if st.session_state.history_mm:
        c2.download_button("📄 Export", export_txt(st.session_state.history_mm),
                           "Math_Mastermind_Solutions.txt", "text/plain")
    with st.form("mm_form", clear_on_submit=True):
        q = st.text_area("Math problem:", height=100, key=f"mm_{st.session_state.k_mm}")
        a, b = st.columns([3, 1])
        go = a.form_submit_button("Solve", use_container_width=True)
        lvl = b.selectbox("Level", ["Basic", "Intermediate", "Advanced"], index=1)
        if go:
            if not q.strip():
                st.warning("⚠️ Enter a problem.")
            else:
                with st.spinner("Solving..."):
                    ans = math_answer(q.strip(), lvl)
                st.session_state.history_mm.insert(0, {"question": q.strip(), "answer": ans, "difficulty": lvl})
                st.session_state.k_mm += 1; st.rerun()

    if not st.session_state.history_mm: return
    with st.container(height=600):
        for i, qa in enumerate(st.session_state.history_mm, 1):
            with st.container(border=True):
                st.markdown(f"**Q{i}:** {esc_md(qa['question'])}  `{qa['difficulty']}`")
                render_answer(qa["answer"])

# ---------------- Safe AI Image Generator ----------------
LOCAL_IMG_MODEL = "stabilityai/sdxl-turbo"  # free, runs on Colab's T4 GPU, no API key
STYLES = {
    "None": "",
    "Photorealistic": ", photorealistic, high detail, natural lighting",
    "Digital art": ", digital art, vibrant colors, detailed",
    "Cartoon": ", cartoon style, friendly, bold outlines",
    "Watercolor": ", watercolor painting, soft colors",
    "3D render": ", 3D render, soft studio lighting",
}

BLOCKED_RE = re.compile(
    r"\b(nude|naked|nsfw|porn\w*|sexual\w*|erotic\w*|fetish|gore|gory|mutilat\w*|dismember\w*|"
    r"behead\w*|tortur\w*|suicide|self[- ]harm|bomb[- ]making|terroris\w*|child abuse|loli\w*)\b",
    re.IGNORECASE,
)

MOD_PROMPT = """You are a strict content-safety classifier for an image generator used in a school setting.
Reply with exactly one word: SAFE or UNSAFE.
UNSAFE means the prompt asks for: sexual or nude content, graphic violence or gore, self-harm,
hate or harassment, illegal activity, weapons instructions, real-person deepfakes, or anything sexualizing minors.
Prompt: """

def check_prompt_safety(prompt):
    """Returns (allowed: bool, reason: str)."""
    if BLOCKED_RE.search(prompt):
        return False, "Your prompt contains a blocked term. Please describe something family-friendly."
    verdict = generate_response(MOD_PROMPT + prompt, temperature=0.0, max_tokens=5).strip().upper()
    if verdict.startswith("UNSAFE"):
        return False, "This prompt was flagged as unsafe by the content filter."
    if verdict.startswith("SAFE"):
        return True, ""
    # LLM moderation unavailable (e.g. API error) -> keyword filter only
    return True, "⚠️ AI moderation unavailable; only the keyword filter was applied."

@st.cache_resource(show_spinner="Loading image model (first run takes a few minutes)...")
def load_local_pipe():
    import torch
    if not torch.cuda.is_available():
        return None
    from diffusers import AutoPipelineForText2Image
    pipe = AutoPipelineForText2Image.from_pretrained(
        LOCAL_IMG_MODEL, torch_dtype=torch.float16, variant="fp16")
    return pipe.to("cuda")

def make_image(prompt):
    errs = []
    # 1) Free local generation on the Colab GPU
    try:
        pipe = load_local_pipe()
        if pipe is not None:
            img = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0,
                       width=512, height=512).images[0]
            buf = io.BytesIO(); img.save(buf, format="PNG")
            return buf.getvalue(), LOCAL_IMG_MODEL
        errs.append("No GPU found (Runtime > Change runtime type > T4 GPU).")
    except Exception as e:
        errs.append(f"local: {type(e).__name__}: {e}")
    # 2) Free web fallback (no key)
    try:
        import requests, urllib.parse
        url = "https://image.pollinations.ai/prompt/" + urllib.parse.quote(prompt)
        r = requests.get(url, params={"width": 768, "height": 768, "nologo": "true", "model": "flux"}, timeout=120)
        if r.ok and r.headers.get("content-type", "").startswith("image"):
            return r.content, "pollinations/flux"
        errs.append(f"pollinations: HTTP {r.status_code}")
    except Exception as e:
        errs.append(f"pollinations: {type(e).__name__}: {e}")
    return None, "Image generation failed.\n" + "\n".join(errs)

def run_safe_ai_image_generator():
    st.title("🎨 Safe AI Image Generator")
    st.session_state.setdefault("history_img", [])
    c1, _ = st.columns([1, 2])
    if c1.button("🧹 Clear", key="c_img"):
        st.session_state.history_img = []; st.rerun()

    prompt = st.text_area("Describe the image:", height=100, key="p_img",
                          placeholder="A friendly robot teaching math in a sunny classroom")
    style = st.selectbox("Style", list(STYLES.keys()))

    if st.button("Generate", key="g_img"):
        p = prompt.strip()
        if not p:
            st.warning("⚠️ Enter a description.")
        else:
            with st.spinner("Checking prompt safety..."):
                ok, msg = check_prompt_safety(p)
            if not ok:
                st.error(f"🚫 {msg}")
            else:
                if msg: st.caption(msg)
                with st.spinner("Generating image..."):
                    data, info = make_image(p + STYLES[style])
                if data is None:
                    st.error(info)
                else:
                    st.session_state.history_img.insert(0, {"prompt": p, "style": style, "png": data, "model": info})

    for i, item in enumerate(st.session_state.history_img):
        st.markdown("---")
        st.image(item["png"], caption=f'{item["prompt"]}  ({item["style"]} · {item["model"].split("/")[-1]})')
        st.download_button("⬇️ Download PNG", item["png"], f"image_{len(st.session_state.history_img) - i}.png",
                           "image/png", key=f"dl_{i}")

def main():
    st.sidebar.title("Choose AI Feature")
    opt = st.sidebar.selectbox("Feature", ["AI Teaching Assistant", "Math Mastermind", "Safe AI Image Generator"],
                               label_visibility="collapsed")
    st.sidebar.selectbox("LLM Provider", ["Groq", "Hugging Face"], key="provider")
    if opt == "AI Teaching Assistant": run_ai_teaching_assistant()
    elif opt == "Math Mastermind": run_math_mastermind()
    else: run_safe_ai_image_generator()

main()
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(APP_CODE)

# ---- Launch Streamlit ----
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true", "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    env=ENV, stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(4)

# ---- Public URL via Cloudflare quick tunnel ----
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod("cloudflared", 0o755)

subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 40:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0); break
print("\n🚀 Open your app here:", url or "Tunnel failed - check network settings / rerun the cell")



🚀 Open your app here: https://riding-hoped-store-meeting.trycloudflare.com
